# Inheritance and exceptions

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*. Original material (2020) in collaboration with
*G. Martínez-Lema* and *M. Kekic*.

**Objectives**

* derive a class from another one: inherit, override, `super()`;
* define an **interface** with an abstract base class;
* decide between **inheritance** ("is a") and **composition** ("has a");
* understand duck typing and the method resolution order;
* use exceptions properly: they are a class hierarchy too.

In [ ]:
import math
from abc import ABC, abstractmethod

## 1. Inheritance

A class can be built on another one. The **derived** (child) class **inherits** all the
attributes and methods of the **base** (parent) class, and can **add** new ones or
**override** (redefine) the inherited ones.

<img src="./img/classes_inheritance.jpeg" width="420">

A car *is a* wheeled vehicle, which *is a* vehicle: everything that can be done with a
vehicle can be done with a car.

### 1.1 An interface: the shapes

A **base class** can define what its derived classes must be able to do, without saying
how. With `ABC` and `@abstractmethod`, Python enforces it: a class with unimplemented
abstract methods cannot be instantiated.

In [ ]:
class Shape(ABC):
    """Interface of the 2D shapes."""

    @abstractmethod
    def area(self):
        """Return the area."""

    @abstractmethod
    def perimeter(self):
        """Return the perimeter."""

    def describe(self):
        """A concrete method, written with the abstract ones."""
        return f'{type(self).__name__}: area {self.area():.3f}, perimeter {self.perimeter():.3f}'

`describe` is **concrete**: it works for every shape, present or future, because it only
uses the interface.

In [ ]:
class Disk(Shape):

    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius


class Rectangle(Shape):

    def __init__(self, width, height):
        self.width, self.height = width, height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)

In [ ]:
for shape in [Disk(1.), Rectangle(2., 3.)]:
    print(shape.describe())

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Abstract classes</b></p>

What happens with each of these? **When** is the problem reported?

```python
Shape()

class Triangle(Shape):
    def __init__(self, a, b, c):
        self.a, self.b, self.c = a, b, c
    def perimeter(self):
        return self.a + self.b + self.c

Triangle(3., 4., 5.)
```

</div>

In [ ]:
class Triangle(Shape):
    def __init__(self, a, b, c):
        self.a, self.b, self.c = a, b, c
    def perimeter(self):
        return self.a + self.b + self.c

Triangle(3., 4., 5.)

<details>
<summary><b>Solution</b></summary>

Both fail with `TypeError: Can't instantiate abstract class ...` (the cell below only
runs the second one). `Shape` is abstract,
and `Triangle` forgot `area`. The error appears when the object is **created**, with a
clear message. Without `ABC`, a forgotten method would appear much later, in the middle of
a computation, or not at all: a base `area` that returns `None` would make
`sum(s.area() for s in shapes)` fail far from the cause.

</details>

### 1.2 A class derived from a derived class: super()

A square *is a* rectangle with equal sides. It only needs its own constructor, which
calls the constructor of the base class with `super()`:

In [ ]:
class Square(Rectangle):

    def __init__(self, side):
        super().__init__(side, side)

sq = Square(2.)
print(sq.describe())
print(isinstance(sq, Square), isinstance(sq, Rectangle), isinstance(sq, Shape))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Which method runs?</b></p>

`Square` defines neither `area` nor `describe`. **Decide** in which class Python finds
each method when we call `sq.describe()`, and in what order it looks.

</div>

In [ ]:
print(Square.__mro__)

<details>
<summary><b>Solution</b></summary>

Python looks for a name in the **method resolution order**, `Square.__mro__`: `Square`,
`Rectangle`, `Shape`, `ABC`, `object`. `describe` is found in `Shape`; inside it,
`self.area()` is looked up again starting from `Square`, and found in `Rectangle`.
`type(self).__name__` is `'Square'`: `self` is always the real object.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — The triangle, done right</b></p>

Complete `Triangle(a, b, c)`: the area with Heron's formula,
$A = \sqrt{s(s-a)(s-b)(s-c)}$ with $s$ the semi-perimeter. The constructor must refuse
sides that cannot form a triangle (use `raise ValueError(...)`; exceptions are covered in
section 5). Test it with the triangle $3, 4, 5$.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
class Triangle(Shape):

    def __init__(self, a, b, c):
        if a + b <= c or a + c <= b or b + c <= a:
            raise ValueError(f'{a}, {b}, {c} do not make a triangle')
        self.a, self.b, self.c = a, b, c

    def perimeter(self):
        return self.a + self.b + self.c

    def area(self):
        s = self.perimeter() / 2
        return math.sqrt(s * (s - self.a) * (s - self.b) * (s - self.c))

assert math.isclose(Triangle(3., 4., 5.).area(), 6.)
print(Triangle(3., 4., 5.).describe())
```

</details>

## 2. Polymorphism and duck typing

Code written for the interface works for every shape:

In [ ]:
def total_area(shapes):
    return sum(shape.area() for shape in shapes)

print(total_area([Disk(1.), Rectangle(2., 3.), Square(2.)]))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Without the base class</b></p>

This class does **not** derive from `Shape`. **Decide**: does
`total_area([Disk(1.), Ring(1., 2.)])` work?

```python
class Ring:
    def __init__(self, r1, r2):
        self.r1, self.r2 = r1, r2
    def area(self):
        return math.pi * (self.r2 ** 2 - self.r1 ** 2)
```

</div>

In [ ]:
class Ring:
    def __init__(self, r1, r2):
        self.r1, self.r2 = r1, r2
    def area(self):
        return math.pi * (self.r2 ** 2 - self.r1 ** 2)

print(total_area([Disk(1.), Ring(1., 2.)]))

<details>
<summary><b>Solution</b></summary>

It works. `total_area` only calls `area()`, and `Ring` has it: Python does not check the
class, only that the method exists (**duck typing**: if it quacks like a duck…). But
`Ring(1., 2.).describe()` fails with `AttributeError`: `describe` lives in `Shape`, and
`Ring` does not inherit it.

</details>

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Then why inherit?</b></p>

If duck typing is enough, why write `Shape` at all? Think of: documentation, errors at
creation time, shared concrete methods like `describe`, `isinstance` checks. In Java,
and for virtual calls in C++, run-time polymorphism requires inheritance (or an
interface); in Python it does not. When would you skip it?

</div>

## 3. Extending a built-in

We can derive from built-in classes too. A list that can give its descending order:

In [ ]:
class MyList(list):

    def descending(self):
        return sorted(self, reverse=True)

xs = MyList([3, 1, 2])
print(xs, xs.descending(), len(xs), type(xs))

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — What type?</b></p>

What is the type of `xs + xs`, and of `xs[:2]`? Does `(xs + xs).descending()` work?

</div>

In [ ]:
print(type(xs + xs), type(xs[:2]))

<details>
<summary><b>Solution</b></summary>

Both are plain `list`, so `(xs + xs).descending()` fails. The methods of a built-in
**hard-code** the built-in type: they create plain `list` objects, not `MyList`. To keep
the type you must override them all, which is long and fragile (`collections.UserList`
helps). This is one reason to prefer composition (next
section), or a plain function `descending(xs)`.

</details>

## 4. Inheritance or composition?

* **Inheritance** says "**is a**": a `Square` is a `Rectangle`.
* **Composition** says "**has a**": a `Detector` has a `Geometry` and a list of `Channel`s,
  stored as attributes.

Composition is more flexible: the parts can be changed independently. The usual advice:
**prefer composition**, and use inheritance for real "is a" relations, one or two levels
deep.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Is a, or has a?</b></p>

For each pair, **decide**: inheritance, composition, or neither?

1. `Muon` and `Particle`
2. `Event` and `Particle`
3. `Calorimeter` and `Detector`
4. `Track` and `Hit`
5. `Histogram2D` and `Histogram1D`

</div>

<details>
<summary><b>Solution</b></summary>

1. Inheritance, if at all: a muon *is a* particle. (Often a `Particle` with a `name`
   attribute is enough: one class, many data.)
2. Composition: an event *has* particles.
3. Inheritance: a calorimeter *is a* detector, with the same interface (`read`,
   `calibrate`…).
4. Composition: a track *has* hits.
5. Neither: a 2D histogram is not a 1D histogram (what would `fill(x)` mean?), and it
   does not contain one. Similar names do not make an "is a".

</details>

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Is a square a rectangle?</b></p>

Suppose `Rectangle` gets a method `set_width(w)` that changes the width only. What happens
when it is called on a `Square`? Is `Square` still a valid substitute for a `Rectangle`?
What does this say about "is a" in code versus in geometry?

</div>

<details>
<summary><b>Solution</b></summary>

`Square(2.).set_width(3.)` gives a "square" of $3 \times 2$: the derived class breaks its
own invariant. In geometry a square is a rectangle; in code, *a mutable square is not a
mutable rectangle*, because it cannot do everything a rectangle can do (change one side).
The rule (Liskov substitution principle): a derived object must work **everywhere** the
base object works, and keep its guarantees (after `set_width(w)`, `area == w * height`).
Our shapes only avoid the problem by convention (nothing stops `sq.width = 3.`); truly
immutable objects, or read-only properties, avoid it.

</details>

## 5. Exceptions

### 5.1 Exceptions are classes

Errors are objects of exception classes, organised in a hierarchy. `ZeroDivisionError`
*is an* `ArithmeticError`, which *is an* `Exception`:

In [ ]:
print(ZeroDivisionError.__mro__)

`KeyboardInterrupt` (Ctrl-C) derives from `BaseException`, not from `Exception`:
`except Exception` does not catch it.

The full tree is in the
[documentation](https://docs.python.org/3/library/exceptions.html#exception-hierarchy).
An `except` clause catches the class it names **and all its derived classes**:

In [ ]:
try:
    1. / 0.
except ArithmeticError as e:
    print('caught:', type(e).__name__, '-', e)

The full form: `try` runs the code; the **first** `except E` that matches the error runs;
`else` runs if there was **no** error; `finally` runs **always** (to close files, release
resources).

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Which except?</b></p>

**Decide** what this prints.

```python
try:
    math.sqrt(-1.)
except ZeroDivisionError:
    print('A')
except ValueError:
    print('B')
except Exception:
    print('C')
else:
    print('D')
finally:
    print('E')
```

</div>

In [ ]:
try:
    math.sqrt(-1.)
except ZeroDivisionError:
    print('A')
except ValueError:
    print('B')
except Exception:
    print('C')
else:
    print('D')
finally:
    print('E')

<details>
<summary><b>Solution</b></summary>

`B` and `E`. `math.sqrt(-1.)` raises `ValueError`; the **first** matching `except` runs,
and only that one; `else` is skipped; `finally` runs. Order matters: put the derived
classes before their base; `except Exception` first would shadow the `ValueError`.

</details>

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — A silent handler</b></p>

This code is meant to handle a zero denominator. What does it print? Is that what you
expected?

```python
x = 2.
try:
    result = 1. / x
    print(reslt)
except:
    print('x is zero')
```

</div>

In [ ]:
x = 2.
try:
    result = 1. / x
    print(reslt)
except:
    print('x is zero')

<details>
<summary><b>Solution</b></summary>

It prints `x is zero`, but `x` is 2. The real error is a typo, `reslt`, a `NameError`,
and the bare `except:` catches **everything** and hides it (it would even catch a
`Ctrl-C`). Always name the exceptions you expect, `except ZeroDivisionError:`, and keep
the `try` block as short as possible.

</details>

### 5.2 Your own exceptions

Derive from the closest built-in exception, so that users can catch it either by its own
name or by its base class:

In [ ]:
class FitError(RuntimeError):
    """The fit failed."""

def fit(data):
    if len(data) < 3:
        raise FitError(f'need at least 3 points, got {len(data)}')
    return sum(data) / len(data)

fit([1., 2.])

In [ ]:
try:
    fit([1., 2.])
except RuntimeError as e:
    print('caught a', type(e).__name__, ':', e)

Raise early, with a message that says what went wrong and with which value. Catch only
where you can **do** something about it; otherwise let it go up.

## 6. Practice

Two exercises with inheritance on real code: a class from the repository, and a family
of probability distributions.

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Vector3D from Vector</b></p>

The file [vector.py](./vector.py) has a class `Vector` of $n$ components, with `+`,
multiplication by a number, indexing and `==`.

1. Read it (`from vector import Vector`). Find at least one bug in `__eq__`: try
   `Vector([1, 2]) == Vector([1, 2, 3])`.
2. Write `Vector3D(Vector)` with a constructor `Vector3D(x, y, z)` and a method `cross`.
3. What is the type of `Vector3D(1, 0, 0) + Vector3D(0, 1, 0)`? Why? Can `cross` be
   applied to it?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
from vector import Vector

print(Vector([1, 2]) == Vector([1, 2, 3]))      # True! zip stops at the shortest

class Vector3D(Vector):
    """A vector of three components."""

    def __init__(self, x, y, z):
        super().__init__([x, y, z])

    def cross(self, other):
        a, b = self.values, other.values
        return Vector3D(a[1] * b[2] - a[2] * b[1],
                        a[2] * b[0] - a[0] * b[2],
                        a[0] * b[1] - a[1] * b[0])

ex, ey = Vector3D(1, 0, 0), Vector3D(0, 1, 0)
print(ex.cross(ey))
print(type(ex + ey))                             # Vector, not Vector3D
```

`__eq__` must first compare the lengths. And `Vector.__add__` builds a `Vector`, so the
sum loses `cross`, as `MyList` lost `descending`. `type(self)(...)` does not help either:
the constructor of `Vector3D` takes three numbers, not a list. Either override the
operators in `Vector3D`, or make all constructors take the same arguments.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[&] Collaborate — A family of distributions</b></p>

A base class `Distribution(ABC)` with an abstract `pdf(x)`, a range `(xmin, xmax)`, and
two **concrete** methods written only with `pdf`: `norm()` (the integral of the pdf, which
must be 1) and `mean()`, both by numerical integration.

In a group: one member writes the base class; the others write `Uniform(a, b)`,
`Exponential(tau)` and `Gaussian(mu, sigma)`, each with a test comparing `norm()` and
`mean()` with the exact values. Then put it all together: does every test pass with
every class?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
class Distribution(ABC):

    def __init__(self, xmin, xmax):
        self.xmin, self.xmax = xmin, xmax

    @abstractmethod
    def pdf(self, x):
        """Probability density at x."""

    def _integrate(self, f, n=10_000):
        h = (self.xmax - self.xmin) / n
        return h * sum(f(self.xmin + (i + 0.5) * h) for i in range(n))

    def norm(self):
        return self._integrate(self.pdf)

    def mean(self):
        return self._integrate(lambda x: x * self.pdf(x))


class Exponential(Distribution):

    def __init__(self, tau):
        super().__init__(0., 30. * tau)
        self.tau = tau

    def pdf(self, x):
        return math.exp(-x / self.tau) / self.tau


class Gaussian(Distribution):

    def __init__(self, mu, sigma):
        super().__init__(mu - 10. * sigma, mu + 10. * sigma)
        self.mu, self.sigma = mu, sigma

    def pdf(self, x):
        z = (x - self.mu) / self.sigma
        return math.exp(-0.5 * z * z) / (math.sqrt(2. * math.pi) * self.sigma)

class Uniform(Distribution):

    def __init__(self, a, b):
        super().__init__(a, b)

    def pdf(self, x):
        return 1. / (self.xmax - self.xmin)

for d, mean in [(Uniform(-1., 3.), 1.), (Exponential(2.), 2.),
                (Gaussian(1., 0.5), 1.), (Gaussian(0., 1.), 0.)]:
    assert math.isclose(d.norm(), 1., rel_tol=1e-5)
    assert math.isclose(d.mean(), mean, rel_tol=1e-5, abs_tol=1e-9)   # abs_tol: mean 0
```

The range is a choice of each derived class (an exponential has no upper limit: we cut
at $30\tau$), and the integration lives once, in the base class. This is the same pattern as
`Shape.describe`, and the one that `scipy.stats` follows.

</details>

## 7. Summary

* Inherit for "is a", one or two levels deep; compose for "has a".
* An abstract base class documents and enforces an interface; duck typing does not need it.
* Exceptions: name the class you expect, keep the `try` short, raise early.

<details>
<summary><b>[i] Cheat sheet — Inheritance and exceptions</b></summary>

| | |
|:--|:--|
| `class B(A):` | `B` inherits from `A`: attributes and methods |
| override | redefine an inherited method in `B` |
| `super().__init__(...)` | call the base class version |
| `class A(ABC)` + `@abstractmethod` | an interface: cannot be instantiated, derived classes must implement |
| `C.__mro__` | the order in which methods are looked up |
| duck typing | Python only needs the method to exist |
| "is a" / "has a" | inheritance / composition: **prefer composition** |
| `try / except E / else / finally` | first matching `except`; `else` if no error; `finally` always |
| `class MyError(ValueError): ...` | your own exception, under the closest built-in |

**Traps**: inherited methods that create objects of the base class; a derived class
that cannot keep the promises of its base; bare `except:`.

</details>

### Check yourself

1. What does an abstract method force, and when is the error raised?
2. Why does `total_area` work with `Ring`, which does not derive from `Shape`?
3. Give an example of "is a" and one of "has a" from a physics code.
4. `except ArithmeticError` catches a `ZeroDivisionError`. Why?
5. Why is `except:` without a class a bad idea?

<details>
<summary><b>Solutions</b></summary>

1. Every concrete derived class must implement it; otherwise creating an object raises a
   `TypeError`.
2. Duck typing: `total_area` only calls `area()`, and `Ring` has it.
3. A `Calorimeter` is a `Detector`; an `Event` has `Track`s.
4. `ZeroDivisionError` derives from `ArithmeticError`: an `except` catches the class and
   all its derived classes.
5. It catches everything, also the errors you did not expect (typos, `Ctrl-C`), and
   hides them.

</details>